
ВКР: Тензорные разложения в теории кооперативных игр
      для стохастического моделирования совместимости игроков  
============================================================

Цель: предсказать эффективность пятёрки хоккеистов NHL
      с помощью Tensor Train (TT) разложения.

Данные: play-by-play статистика NHL (сезоны 2017/18 — 2022/23)

Разделение данных:
  Train: сезоны 2017/18, 2018/19, 2019/20, 2020/21
  Valid: сезон 2021/22
  Test:  сезон 2022/23

Метрика: бинарная классификация

  - **y = 1, если пятёрка забила больше голов, чем пропустила**
  - **y = 0, иначе**


In [1]:
!pip install teneva

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
# Импорт библиотек
# -------------------------------------------------------------



import pandas as pd
import numpy as np
import os
from collections import defaultdict
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import reverse_cuthill_mckee
import teneva

## 1. Обработка данных

### Исходные данные
Файлы play-by-play (pbp) содержат события каждого матча NHL:
вбрасывания (FAC), остановки (STOP), составы пятёрок на льду,
а также счёт матча (Home_Score, Away_Score).

### Алгоритм обработки

**Шаг 1. Извлечение пятёрок из матчей:**
- Для каждого матча находим все вбрасывания (FAC)
- Между двумя последовательными вбрасываниями — один интервал
- На каждом интервале фиксируем составы домашней и гостевой пятёрок
  (по 5 полевых игроков)
- Для каждой пятёрки считаем: время на льду, голы забитые (GF),
  голы пропущенные (GA)

**Шаг 2. Агрегация по уникальным пятёркам:**
- Одна и та же пятёрка может встречаться в разных матчах и сезонах
- Суммируем GF (забитые), GA (пропущенные), время по всем появлениям
- Более поздним сезонам присваиваем больший вес
  (2017/18→1, 2018/19→2, 2019/20→3, 2020/21→4)

**Шаг 3. Фильтрация:**
- Минимум 5 появлений (n_games ≥ 5) — отсекаем случайные пятёрки
- Минимум 120 секунд суммарного времени на льду
- Хотя бы один гол (GF + GA > 0) — иначе нельзя определить метрику

**Шаг 4. Бинарная метрика:**
- y = 1, если GF > GA (пятёрка забила больше, чем пропустила)
- y = 0, иначе

**Шаг 5. Построение матрицы для тензора:**
- Строки — уникальные пятёрки (наблюдения)
- Столбцы — все игроки (оси тензора)
- Значение I[i,j] = 1, если j-й игрок входит в i-ю пятёрку, иначе 0
- В каждой строке ровно 5 единиц

**Шаг 6. Переупорядочение столбцов:**
- Игроки, часто играющие вместе, размещаются рядом, чтобы улучшить качество TT-разложения, так как соседние ядра взаимодействуют сильнее

## Схема разделения данных

Используется хронологическое разделение (без утечки данных из будущего):

**Этап 1 — подбор гиперпараметров:**
- Train: сезоны 2017/18, 2018/19, 2019/20, 2020/21
- Valid: сезон 2021/22
- Подбираем: ранг TT-разложения, порядок ANOVA, параметр регуляризации

**Этап 2 — финальная модель:**
- Train: сезоны 2017/18 — 2021/22 (включая бывший valid)
- Test: сезон 2022/23
- Оценка на тесте проводится однократно

Веса сезонов: более поздние сезоны получают больший вес,
т.к. текущая форма игроков важнее исторической.

In [3]:
# ============================================================
# Параметры
# ============================================================

pbp_dir = 'data_nhl'

train_seasons_list = ['20172018', '20182019', '20192020', '20202021']
valid_seasons_list = ['20212022']
test_seasons_list  = ['20222023']

# Веса сезонов: чем новее, тем важнее
season_weights = {
    '20172018': 1,
    '20182019': 2,
    '20192020': 3,
    '20202021': 4,
    '20212022': 5,
    '20222023': 5,
}

# Фильтрация
MIN_TIME = 120.0    # минимум секунд на льду
MIN_GAMES = 5       # минимум появлений пятёрки

In [4]:
# ============================================================
# Функция обработки одного матча
# ============================================================

def process_game(pbp_df, game_id):
    """
    Обрабатывает один матч.
    
    Алгоритм:
    1. Находим все вбрасывания (FAC) — они делят матч на интервалы
    2. На каждом интервале определяем составы пятёрок (home/away)
    3. Считаем голы за/против для каждой пятёрки на каждом интервале
    
    Возвращает список словарей:
        {'players': tuple, 'time': float, 'gf': int, 'ga': int}
    """
    game = pbp_df[pbp_df['Game_Id'] == game_id].copy()
    if game.empty:
        return []
    
    game = game.sort_values('Seconds_Elapsed').reset_index(drop=True)
    
    # Парсим счёт
    for col in ['Home_Score', 'Away_Score']:
        game[col] = pd.to_numeric(game[col], errors='coerce').fillna(0).astype(int)
    
    game['Home_Score_prev'] = game['Home_Score'].shift(1).fillna(0).astype(int)
    game['Away_Score_prev'] = game['Away_Score'].shift(1).fillna(0).astype(int)
    
    # Определяем моменты голов
    home_goal = game['Home_Score'] > game['Home_Score_prev']
    away_goal = game['Away_Score'] > game['Away_Score_prev']
    goals = game[home_goal | away_goal].copy()
    
    # Вбрасывания — границы интервалов
    facs = game[game['Event'] == 'FAC'].sort_values('Seconds_Elapsed').reset_index(drop=True)
    if facs.empty:
        return []
    
    home_cols = [f'homePlayer{i}_id' for i in range(1, 6)]
    away_cols = [f'awayPlayer{i}_id' for i in range(1, 6)]
    
    # Строим интервалы
    intervals = []
    for idx in range(len(facs)):
        row = facs.iloc[idx]
        t_start = row['Seconds_Elapsed']
        t_end = facs.iloc[idx + 1]['Seconds_Elapsed'] if idx + 1 < len(facs) else 3900.0
        
        home_five = tuple(sorted([int(row[c]) for c in home_cols if pd.notna(row[c])]))
        away_five = tuple(sorted([int(row[c]) for c in away_cols if pd.notna(row[c])]))
        
        if len(home_five) == 5 and len(away_five) == 5:
            intervals.append({
                'start': t_start, 'end': t_end,
                'home': home_five, 'away': away_five
            })
    
    # Считаем статистику по пятёркам
    ice_time = defaultdict(float)
    goals_for = defaultdict(int)
    goals_against = defaultdict(int)
    
    for interval in intervals:
        dur = interval['end'] - interval['start']
        ice_time[interval['home']] += dur
        ice_time[interval['away']] += dur
        
        for _, g in goals.iterrows():
            t = g['Seconds_Elapsed']
            if interval['start'] <= t < interval['end']:
                if g['Home_Score'] > g['Home_Score_prev']:
                    goals_for[interval['home']] += 1
                    goals_against[interval['away']] += 1
                else:
                    goals_for[interval['away']] += 1
                    goals_against[interval['home']] += 1
    
    result = []
    for five in ice_time.keys():
        result.append({
            'players': five,
            'time': ice_time[five],
            'gf': goals_for[five],
            'ga': goals_against[five],
        })
    return result

In [5]:
# ============================================================
# Загрузка и обработка матчей
# ============================================================

all_seasons = train_seasons_list + valid_seasons_list
raw_data = []

for year in all_seasons:
    pbp_file = os.path.join(pbp_dir, f'nhl_pbp{year}.csv')
    
    if not os.path.exists(pbp_file):
        print(f"⚠️ {pbp_file} не найден!")
        continue
    
    print(f"Обработка {year}...", end=' ')
    pbp = pd.read_csv(
        pbp_file, sep=',', quotechar='"',
        escapechar='\\', engine='python', on_bad_lines='skip'
    )
    
    game_ids = pbp['Game_Id'].unique()
    count, errors = 0, 0
    
    for game_id in game_ids:
        try:
            game_data = process_game(pbp, game_id)
            for row in game_data:
                row['season'] = year
                row['weight'] = season_weights[year]
            raw_data.extend(game_data)
            count += len(game_data)
        except:
            errors += 1
    
    print(f"Игр: {len(game_ids)}, пятёрок: {count}, ошибок: {errors}")

print(f"\n✅ Всего сырых наблюдений: {len(raw_data)}")

Обработка 20172018... Игр: 1355, пятёрок: 72915, ошибок: 0
Игр: 1358, пятёрок: 72254, ошибок: 0
Игр: 1201, пятёрок: 63138, ошибок: 0
Игр: 952, пятёрок: 48931, ошибок: 0
Игр: 531, пятёрок: 27726, ошибок: 0

✅ Всего сырых наблюдений: 284964


## Агрегация и фильтрация

Для каждой уникальной пятёрки суммируем голы и время по всем матчам,
взвешивая по сезону (новые сезоны важнее).

Фильтрация:
- Минимум 5 появлений пятёрки (отсекаем случайные составы)
- Минимум 120 секунд суммарного времени на льду
- Хотя бы один гол (GF + GA > 0)

In [6]:
# ============================================================
# Агрегация по уникальным пятёркам
# ============================================================

train_seasons = {'20172018', '20182019', '20192020', '20202021'}
valid_seasons = {'20212022'}

agg_train = defaultdict(lambda: {'gf': 0, 'ga': 0, 'time': 0.0, 'n_games': 0})
agg_valid = defaultdict(lambda: {'gf': 0, 'ga': 0, 'time': 0.0, 'n_games': 0})

for row in raw_data:
    key = row['players']
    season = row['season']
    w = row['weight']
    
    if season in train_seasons:
        target = agg_train[key]
    elif season in valid_seasons:
        target = agg_valid[key]
    else:
        continue
    
    target['gf'] += row['gf'] * w
    target['ga'] += row['ga'] * w
    target['time'] += row['time'] * w
    target['n_games'] += 1

print(f"Уникальных пятёрок в train (до фильтрации): {len(agg_train)}")
print(f"Уникальных пятёрок в valid (до фильтрации): {len(agg_valid)}")

# --- Фильтрация ---
def make_dataset(agg):
    ds = []
    for five, stats in agg.items():
        if stats['time'] < MIN_TIME:
            continue
        if stats['n_games'] < MIN_GAMES:
            continue
        if stats['gf'] + stats['ga'] == 0:
            continue
        y = 1 if stats['gf'] > stats['ga'] else 0
        ds.append({
            'players': five,
            'y': y,
            'n_games': stats['n_games'],
            'gf': stats['gf'],
            'ga': stats['ga'],
        })
    return ds

dataset_train = make_dataset(agg_train)
dataset_valid = make_dataset(agg_valid)

ys_trn = [r['y'] for r in dataset_train]
ys_vld = [r['y'] for r in dataset_valid]

print(f"\nПосле фильтрации:")
print(f"  Train: {len(dataset_train)} пятёрок, y=1: {sum(ys_trn)/len(ys_trn):.3f}")
print(f"  Valid: {len(dataset_valid)} пятёрок, y=1: {sum(ys_vld)/len(ys_vld):.3f}")

n_trn = [r['n_games'] for r in dataset_train]
print(f"  Train игр/пятёрку: mean={np.mean(n_trn):.1f}, median={np.median(n_trn):.0f}, max={np.max(n_trn)}")

Уникальных пятёрок в train (до фильтрации): 125779
Уникальных пятёрок в valid (до фильтрации): 16882

После фильтрации:
  Train: 9172 пятёрок, y=1: 0.481
  Valid: 767 пятёрок, y=1: 0.542
  Train игр/пятёрку: mean=10.1, median=7, max=211


## 1.3 Построение бинарной матрицы

Строим матрицу I размера (n_пятёрок × n_игроков):
- I[i,j] = 1, если j-й игрок входит в i-ю пятёрку
- I[i,j] = 0, иначе
- В каждой строке ровно 5 единиц

Индексация игроков строится **только по train**.
Из valid удаляются пятёрки с игроками, не встречавшимися в train.

Столбцы переупорядочиваются, чтобы игроки, часто играющие вместе, располагались рядом.

In [7]:
# ============================================================
# Построение бинарной матрицы
# ============================================================

# --- Индексация игроков (только по train) ---
player_freq = defaultdict(int)
for row in dataset_train:
    for p in row['players']:
        player_freq[p] += 1

sorted_players = sorted(player_freq.keys(), key=lambda p: -player_freq[p])
player_to_idx = {pid: idx for idx, pid in enumerate(sorted_players)}
n_players = len(player_to_idx)

print(f"Уникальных игроков в train: {n_players}")
print(f"Топ-5 по частоте:")
for i in range(5):
    pid = sorted_players[i]
    print(f"  id={pid}, в {player_freq[pid]} пятёрках")

# --- Переупорядочение (Reverse Cuthill-McKee) ---
co_occ = np.zeros((n_players, n_players), dtype=int)
for row in dataset_train:
    idxs = [player_to_idx[p] for p in row['players']]
    for i in range(5):
        for j in range(i+1, 5):
            co_occ[idxs[i], idxs[j]] += 1
            co_occ[idxs[j], idxs[i]] += 1

rcm_order = reverse_cuthill_mckee(csr_matrix(co_occ))
print(f"\n Переупорядочение выполнено")

# --- Бинарная матрица: TRAIN ---
I_trn = np.zeros((len(dataset_train), n_players), dtype=int)
y_trn = np.zeros(len(dataset_train), dtype=float)

for i, row in enumerate(dataset_train):
    for p in row['players']:
        I_trn[i, player_to_idx[p]] = 1
    y_trn[i] = row['y']

I_trn = I_trn[:, rcm_order]

# --- Бинарная матрица: VALID ---
dataset_valid_clean = []
for row in dataset_valid:
    all_known = True
    for p in row['players']:
        if p not in player_to_idx:
            all_known = False
            break
    if all_known:
        dataset_valid_clean.append(row)

print(f"\nValid до фильтрации по игрокам: {len(dataset_valid)}")
print(f"Valid после фильтрации: {len(dataset_valid_clean)}")

I_vld = np.zeros((len(dataset_valid_clean), n_players), dtype=int)
y_vld = np.zeros(len(dataset_valid_clean), dtype=float)

for i, row in enumerate(dataset_valid_clean):
    for p in row['players']:
        I_vld[i, player_to_idx[p]] = 1
    y_vld[i] = row['y']

I_vld = I_vld[:, rcm_order]

# --- Убираем столбцы, где в train только нули ---
good_cols = [k for k in range(I_trn.shape[1]) if I_trn[:, k].max() == 1]
bad_cols = [k for k in range(I_trn.shape[1]) if I_trn[:, k].max() == 0]

if len(bad_cols) > 0:
    mask_vld = np.ones(len(I_vld), dtype=bool)
    for k in bad_cols:
        mask_vld &= (I_vld[:, k] == 0)
    I_vld = I_vld[mask_vld]
    y_vld = y_vld[mask_vld]

I_trn = I_trn[:, good_cols]
I_vld = I_vld[:, good_cols]

# --- Итог ---
print(f"\n{'='*50}")
print(f"Финальные данные для обучения:")
print(f"  I_trn: {I_trn.shape} (пятёрок × игроков)")
print(f"  I_vld: {I_vld.shape}")
print(f"  Убрано столбцов: {len(bad_cols)}")
print(f"  Train y=1: {y_trn.mean():.3f}")
print(f"  Valid y=1: {y_vld.mean():.3f}")
print(f"  Сумма по строке (проверка): {I_trn[0].sum()}")
print(f"{'='*50}")

Уникальных игроков в train: 1126
Топ-5 по частоте:
  id=8475167, в 168 пятёрках
  id=8476902, в 165 пятёрках
  id=8470613, в 156 пятёрках
  id=8474565, в 152 пятёрках
  id=8475158, в 152 пятёрках

 Переупорядочение выполнено

Valid до фильтрации по игрокам: 767
Valid после фильтрации: 623

Финальные данные для обучения:
  I_trn: (9172, 1126) (пятёрок × игроков)
  I_vld: (623, 1126)
  Убрано столбцов: 0
  Train y=1: 0.481
  Valid y=1: 0.567
  Сумма по строке (проверка): 5


## 2. Построение тензорных моделей

### Tensor Train (TT) разложение

Мы строим тензор T размерности d = n_игроков, где каждая ось 
принимает значения {0, 1} (игрок в пятёрке или нет).

Полный тензор имел бы 2^1126 элементов — это невозможно хранить.
TT-разложение представляет его как произведение малых ядер:

    T[i₁, i₂, ..., i_d] = G₁[i₁] × G₂[i₂] × ... × G_d[i_d]

где G_k — ядро размера (r_{k-1}, 2, r_k), r — TT-ранг.

### Методы обучения

**ANOVA (Analysis of Variance):**
- Аналитический метод, строит TT-тензор за один проход
- order=1: учитывает только индивидуальные вклады игроков
- order=2: добавляет парные взаимодействия (синергия между игроками)
- Стабильный, не переобучается

**ALS (Alternating Least Squares):**
- Итеративный метод, улучшает начальное приближение от ANOVA
- На каждой итерации оптимизирует одно ядро, фиксируя остальные
- Может переобучаться при большом числе осей — требует регуляризации

### Эксперименты

Обучаем несколько моделей с разными гиперпараметрами,
оцениваем на валидации и выбираем лучшую.

In [8]:
# ============================================================
# 2.1 ANOVA: order=1, подбор ранга
# ============================================================

print("=" * 60)
print("Эксперимент 1: ANOVA order=1 (индивидуальные вклады)")
print("=" * 60)

results = []

for r in [2, 3, 5, 7]:
    Y = teneva.anova(I_trn, y_trn, r=r, order=1)
    
    y_pred_trn = teneva.get_many(Y, I_trn)
    y_pred_vld = teneva.get_many(Y, I_vld)
    
    trn_acc = np.mean((y_pred_trn > 0.5) == y_trn)
    vld_acc = np.mean((y_pred_vld > 0.5) == y_vld)
    trn_mae = np.mean(np.abs(y_pred_trn - y_trn))
    vld_mae = np.mean(np.abs(y_pred_vld - y_vld))
    
    results.append({
        'method': 'ANOVA',
        'order': 1,
        'rank': r,
        'train_acc': round(trn_acc, 4),
        'valid_acc': round(vld_acc, 4),
        'train_mae': round(trn_mae, 4),
        'valid_mae': round(vld_mae, 4),
    })
    
    print(f"  r={r}: Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")

# Сохраняем лучшую модель order=1
best_r1 = max(results, key=lambda x: x['valid_acc'])['rank']
Y_best_o1 = teneva.anova(I_trn, y_trn, r=best_r1, order=1)
print(f"\nЛучший ранг (order=1): r={best_r1}")

Эксперимент 1: ANOVA order=1 (индивидуальные вклады)
  r=2: Train Acc=0.6229, Valid Acc=0.4735
  r=3: Train Acc=0.6229, Valid Acc=0.4735
  r=5: Train Acc=0.6229, Valid Acc=0.4735
  r=7: Train Acc=0.6229, Valid Acc=0.4735

Лучший ранг (order=1): r=2


In [9]:
# ============================================================
# 2.2 ANOVA: order=2 (парные взаимодействия)
# ============================================================

# print("=" * 60)
# print("Эксперимент 2: ANOVA order=2 (парные взаимодействия)")
# print("Это может занять несколько минут...")
# print("=" * 60)

# for r in [2, 3, 5]:
#     print(f"  r={r}...", end=' ', flush=True)
    
#     Y = teneva.anova(I_trn, y_trn, r=r, order=2)
    
#     y_pred_trn = teneva.get_many(Y, I_trn)
#     y_pred_vld = teneva.get_many(Y, I_vld)
    
#     trn_acc = np.mean((y_pred_trn > 0.5) == y_trn)
#     vld_acc = np.mean((y_pred_vld > 0.5) == y_vld)
#     trn_mae = np.mean(np.abs(y_pred_trn - y_trn))
#     vld_mae = np.mean(np.abs(y_pred_vld - y_vld))
    
#     results.append({
#         'method': 'ANOVA',
#         'order': 2,
#         'rank': r,
#         'train_acc': round(trn_acc, 4),
#         'valid_acc': round(vld_acc, 4),
#         'train_mae': round(trn_mae, 4),
#         'valid_mae': round(vld_mae, 4),
#     })
    
#     print(f"Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")

In [10]:
# --- Распределение частот игроков ---
freqs = sorted(player_freq.values(), reverse=True)

for threshold in [10, 20, 30, 50, 70, 100]:
    n_players_t = sum(1 for f in freqs if f >= threshold)
    
    # Сколько пятёрок останется (все 5 игроков должны пройти порог)
    n_fives = 0
    for row in dataset_train:
        if all(player_freq[p] >= threshold for p in row['players']):
            n_fives += 1
    
    print(f"Порог >= {threshold:3d}: игроков={n_players_t:4d}, "
          f"пятёрок train={n_fives:5d}, "
          f"пар order=2={n_players_t*(n_players_t-1)//2:>10d}")

Порог >=  10: игроков= 804, пятёрок train= 8038, пар order=2=    322806
Порог >=  20: игроков= 658, пятёрок train= 6536, пар order=2=    216153
Порог >=  30: игроков= 559, пятёрок train= 5020, пар order=2=    155961
Порог >=  50: игроков= 408, пятёрок train= 2518, пар order=2=     83028
Порог >=  70: игроков= 276, пятёрок train=  939, пар order=2=     37950
Порог >= 100: игроков= 103, пятёрок train=   37, пар order=2=      5253


In [11]:
# # ============================================================
# # Попробуем порог >= 20 (больше данных)
# # ============================================================

# FREQ_THRESHOLD = 20

# frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}
# print(f"Игроков с >= {FREQ_THRESHOLD} пятёрками: {len(frequent_players)}")

# dataset_train_f = [
#     row for row in dataset_train
#     if all(p in frequent_players for p in row['players'])
# ]
# dataset_valid_f = [
#     row for row in dataset_valid_clean
#     if all(p in frequent_players for p in row['players'])


# print(f"Train пятёрок: {len(dataset_train_f)}")
# print(f"Valid пятёрок: {len(dataset_valid_f)}")

# # --- Новая индексация ---
# freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
# pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
# n_pl = len(pidx)

# # --- RCM ---
# co = np.zeros((n_pl, n_pl), dtype=int)
# for row in dataset_train_f:
#     idxs = [pidx[p] for p in row['players']]
#     for i in range(5):
#         for j in range(i+1, 5):
#             co[idxs[i], idxs[j]] += 1
#             co[idxs[j], idxs[i]] += 1

# rcm = reverse_cuthill_mckee(csr_matrix(co))

# # --- Бинарные матрицы ---
# I_trn_f = np.zeros((len(dataset_train_f), n_pl), dtype=int)
# y_trn_f = np.zeros(len(dataset_train_f), dtype=float)
# for i, row in enumerate(dataset_train_f):
#     for p in row['players']:
#         I_trn_f[i, pidx[p]] = 1
#     y_trn_f[i] = row['y']
# I_trn_f = I_trn_f[:, rcm]

# I_vld_f = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
# y_vld_f = np.zeros(len(dataset_valid_f), dtype=float)
# for i, row in enumerate(dataset_valid_f):
#     for p in row['players']:
#         I_vld_f[i, pidx[p]] = 1
#     y_vld_f[i] = row['y']
# I_vld_f = I_vld_f[:, rcm]

# # --- Убираем плохие столбцы ---
# good = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 1]
# bad = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 0]

# if len(bad) > 0:
#     mask = np.ones(len(I_vld_f), dtype=bool)
#     for k in bad:
#         mask &= (I_vld_f[:, k] == 0)
#     I_vld_f = I_vld_f[mask]
#     y_vld_f = y_vld_f[mask]

# I_trn_f = I_trn_f[:, good]
# I_vld_f = I_vld_f[:, good]

# print(f"\n{'='*50}")
# print(f"Финальные данные (порог >= {FREQ_THRESHOLD}):")
# print(f"  I_trn: {I_trn_f.shape}")
# print(f"  I_vld: {I_vld_f.shape}")
# print(f"  Train y=1: {y_trn_f.mean():.3f}")
# print(f"  Valid y=1: {y_vld_f.mean():.3f}")
# print(f"{'='*50}")

In [12]:
# # ============================================================
# # ANOVA order=1 (baseline) и order=2 (парные взаимодействия)
# # ============================================================

# import time

# results_f = []

# # --- Order=1 baseline ---
# print("ANOVA order=1...")
# Y1 = teneva.anova(I_trn_f, y_trn_f, r=2, order=1)
# y_p_trn = teneva.get_many(Y1, I_trn_f)
# y_p_vld = teneva.get_many(Y1, I_vld_f)

# trn_acc = np.mean((y_p_trn > 0.5) == y_trn_f)
# vld_acc = np.mean((y_p_vld > 0.5) == y_vld_f)
# print(f"  order=1, r=2: Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")

# results_f.append({
#     'method': 'ANOVA', 'order': 1, 'rank': 2,
#     'train_acc': round(trn_acc, 4), 'valid_acc': round(vld_acc, 4)
# })

# # --- Order=2 ---
# for r in [2, 3, 5]:
#     print(f"\nANOVA order=2, r={r}...", flush=True)
#     t0 = time.time()
    
#     Y2 = teneva.anova(I_trn_f, y_trn_f, r=r, order=2)
    
#     elapsed = time.time() - t0
#     print(f"  Время: {elapsed:.0f} сек")
    
#     y_p_trn = teneva.get_many(Y2, I_trn_f)
#     y_p_vld = teneva.get_many(Y2, I_vld_f)
    
#     trn_acc = np.mean((y_p_trn > 0.5) == y_trn_f)
#     vld_acc = np.mean((y_p_vld > 0.5) == y_vld_f)
#     trn_mae = np.mean(np.abs(y_p_trn - y_trn_f))
#     vld_mae = np.mean(np.abs(y_p_vld - y_vld_f))
    
#     print(f"  Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")
#     print(f"  Train MAE={trn_mae:.4f}, Valid MAE={vld_mae:.4f}")
    
#     results_f.append({
#         'method': 'ANOVA', 'order': 2, 'rank': r,
#         'train_acc': round(trn_acc, 4), 'valid_acc': round(vld_acc, 4)
#     })

# # --- Итог ---
# print(f"\n{'='*60}")
# print("Сводка результатов:")
# print(f"{'='*60}")
# for res in results_f:
#     print(f"  {res['method']} order={res['order']} r={res['rank']}: "
#           f"Train={res['train_acc']:.4f}, Valid={res['valid_acc']:.4f}")

In [13]:
# # ============================================================
# # Перестроение данных с порогом >= 50
# # ============================================================

# FREQ_THRESHOLD = 50

# frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}
# print(f"Игроков с >= {FREQ_THRESHOLD} пятёрками: {len(frequent_players)}")

# dataset_train_f = [
#     row for row in dataset_train
#     if all(p in frequent_players for p in row['players'])
# ]
# dataset_valid_f = [
#     row for row in dataset_valid_clean
#     if all(p in frequent_players for p in row['players'])
# ]

# print(f"Train пятёрок: {len(dataset_train_f)}")
# print(f"Valid пятёрок: {len(dataset_valid_f)}")

# # --- Новая индексация ---
# freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
# pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
# n_pl = len(pidx)

# # --- RCM ---
# co = np.zeros((n_pl, n_pl), dtype=int)
# for row in dataset_train_f:
#     idxs = [pidx[p] for p in row['players']]
#     for i in range(5):
#         for j in range(i+1, 5):
#             co[idxs[i], idxs[j]] += 1
#             co[idxs[j], idxs[i]] += 1

# rcm = reverse_cuthill_mckee(csr_matrix(co))

# # --- Бинарные матрицы ---
# I_trn_f = np.zeros((len(dataset_train_f), n_pl), dtype=int)
# y_trn_f = np.zeros(len(dataset_train_f), dtype=float)
# for i, row in enumerate(dataset_train_f):
#     for p in row['players']:
#         I_trn_f[i, pidx[p]] = 1
#     y_trn_f[i] = row['y']
# I_trn_f = I_trn_f[:, rcm]

# I_vld_f = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
# y_vld_f = np.zeros(len(dataset_valid_f), dtype=float)
# for i, row in enumerate(dataset_valid_f):
#     for p in row['players']:
#         I_vld_f[i, pidx[p]] = 1
#     y_vld_f[i] = row['y']
# I_vld_f = I_vld_f[:, rcm]

# # --- Убираем плохие столбцы ---
# good = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 1]
# bad = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 0]

# if len(bad) > 0:
#     mask = np.ones(len(I_vld_f), dtype=bool)
#     for k in bad:
#         mask &= (I_vld_f[:, k] == 0)
#     I_vld_f = I_vld_f[mask]
#     y_vld_f = y_vld_f[mask]

# I_trn_f = I_trn_f[:, good]
# I_vld_f = I_vld_f[:, good]

# print(f"\n{'='*50}")
# print(f"Финальные данные (порог >= {FREQ_THRESHOLD}):")
# print(f"  I_trn: {I_trn_f.shape}")
# print(f"  I_vld: {I_vld_f.shape}")
# print(f"  Train y=1: {y_trn_f.mean():.3f}")
# print(f"  Valid y=1: {y_vld_f.mean():.3f}")
# print(f"{'='*50}")

# # ============================================================
# # ANOVA order=1 и order=2
# # ============================================================
# import time

# results_f = []

# # --- Order=1 ---
# print("\nANOVA order=1, r=2...")
# Y1 = teneva.anova(I_trn_f, y_trn_f, r=2, order=1)
# y_p_trn = teneva.get_many(Y1, I_trn_f)
# y_p_vld = teneva.get_many(Y1, I_vld_f)
# trn_acc = np.mean((y_p_trn > 0.5) == y_trn_f)
# vld_acc = np.mean((y_p_vld > 0.5) == y_vld_f)
# print(f"  Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")
# results_f.append({'method': 'ANOVA', 'order': 1, 'rank': 2,
#                   'train_acc': round(trn_acc, 4), 'valid_acc': round(vld_acc, 4)})

# # --- Order=2 ---
# for r in [2, 3, 5]:
#     print(f"\nANOVA order=2, r={r}...", flush=True)
#     t0 = time.time()
    
#     Y2 = teneva.anova(I_trn_f, y_trn_f, r=r, order=2)
    
#     elapsed = time.time() - t0
#     print(f"  Время: {elapsed:.0f} сек")
    
#     y_p_trn = teneva.get_many(Y2, I_trn_f)
#     y_p_vld = teneva.get_many(Y2, I_vld_f)
    
#     trn_acc = np.mean((y_p_trn > 0.5) == y_trn_f)
#     vld_acc = np.mean((y_p_vld > 0.5) == y_vld_f)
    
#     print(f"  Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")
    
#     results_f.append({'method': 'ANOVA', 'order': 2, 'rank': r,
#                       'train_acc': round(trn_acc, 4), 'valid_acc': round(vld_acc, 4)})

# # --- Сводка ---
# print(f"\n{'='*60}")
# print("Сводка:")
# for res in results_f:
#     print(f"  {res['method']} order={res['order']} r={res['rank']}: "
#           f"Train={res['train_acc']:.4f}, Valid={res['valid_acc']:.4f}")

Игроков с >= 50 пятёрками: 408
Train пятёрок: 2518
Valid пятёрок: 111

Финальные данные (порог >= 50):
  I_trn: (2518, 407)
  I_vld: (111, 407)
  Train y=1: 0.488
  Valid y=1: 0.577

ANOVA order=1, r=2...
  Train Acc=0.6223, Valid Acc=0.4144

ANOVA order=2, r=2...


KeyboardInterrupt: 

In [14]:
# Проверяем маленькие размерности
for threshold in [70, 80, 90, 100, 120, 150]:
    n_p = sum(1 for f in player_freq.values() if f >= threshold)
    n_f = sum(1 for row in dataset_train
              if all(player_freq[p] >= threshold for p in row['players']))
    pairs = n_p * (n_p - 1) // 2
    print(f"Порог >= {threshold:3d}: игроков={n_p:4d}, "
          f"пятёрок={n_f:5d}, пар={pairs:>8d}")

Порог >=  70: игроков= 276, пятёрок=  939, пар=   37950
Порог >=  80: игроков= 213, пятёрок=  508, пар=   22578
Порог >=  90: игроков= 151, пятёрок=  114, пар=   11325
Порог >= 100: игроков= 103, пятёрок=   37, пар=    5253
Порог >= 120: игроков=  31, пятёрок=    1, пар=     465
Порог >= 150: игроков=   6, пятёрок=    0, пар=      15


In [ ]:
# # ============================================================
# # Порог >= 70: строим данные и пробуем order=2
# # ============================================================
# import time

# FREQ_THRESHOLD = 70

# frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}

# dataset_train_f = [
#     row for row in dataset_train
#     if all(p in frequent_players for p in row['players'])
# ]
# dataset_valid_f = [
#     row for row in dataset_valid_clean
#     if all(p in frequent_players for p in row['players'])
# ]

# print(f"Игроков: {len(frequent_players)}")
# print(f"Train: {len(dataset_train_f)}, Valid: {len(dataset_valid_f)}")

# # --- Индексация + RCM ---
# freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
# pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
# n_pl = len(pidx)

# co = np.zeros((n_pl, n_pl), dtype=int)
# for row in dataset_train_f:
#     idxs = [pidx[p] for p in row['players']]
#     for i in range(5):
#         for j in range(i+1, 5):
#             co[idxs[i], idxs[j]] += 1
#             co[idxs[j], idxs[i]] += 1

# rcm = reverse_cuthill_mckee(csr_matrix(co))

# # --- Бинарные матрицы ---
# I_trn_f = np.zeros((len(dataset_train_f), n_pl), dtype=int)
# y_trn_f = np.zeros(len(dataset_train_f), dtype=float)
# for i, row in enumerate(dataset_train_f):
#     for p in row['players']:
#         I_trn_f[i, pidx[p]] = 1
#     y_trn_f[i] = row['y']
# I_trn_f = I_trn_f[:, rcm]

# I_vld_f = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
# y_vld_f = np.zeros(len(dataset_valid_f), dtype=float)
# for i, row in enumerate(dataset_valid_f):
#     for p in row['players']:
#         I_vld_f[i, pidx[p]] = 1
#     y_vld_f[i] = row['y']
# I_vld_f = I_vld_f[:, rcm]

# good = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 1]
# bad = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 0]
# if len(bad) > 0:
#     mask = np.ones(len(I_vld_f), dtype=bool)
#     for k in bad:
#         mask &= (I_vld_f[:, k] == 0)
#     I_vld_f = I_vld_f[mask]
#     y_vld_f = y_vld_f[mask]
# I_trn_f = I_trn_f[:, good]
# I_vld_f = I_vld_f[:, good]

# print(f"\nI_trn: {I_trn_f.shape}, y=1: {y_trn_f.mean():.3f}")
# print(f"I_vld: {I_vld_f.shape}, y=1: {y_vld_f.mean():.3f}")

# # --- ANOVA order=1 ---
# print(f"\n{'='*50}")
# print("ANOVA order=1, r=2...")
# Y1 = teneva.anova(I_trn_f, y_trn_f, r=2, order=1)
# y_p_t = teneva.get_many(Y1, I_trn_f)
# y_p_v = teneva.get_many(Y1, I_vld_f)
# print(f"  Train Acc={np.mean((y_p_t>0.5)==y_trn_f):.4f}, "
#       f"Valid Acc={np.mean((y_p_v>0.5)==y_vld_f):.4f}")

# # --- ANOVA order=2 ---
# print(f"\nANOVA order=2, r=2...")
# t0 = time.time()
# Y2 = teneva.anova(I_trn_f, y_trn_f, r=2, order=2)
# print(f"  Время: {time.time()-t0:.0f} сек")

# y_p_t = teneva.get_many(Y2, I_trn_f)
# y_p_v = teneva.get_many(Y2, I_vld_f)
# print(f"  Train Acc={np.mean((y_p_t>0.5)==y_trn_f):.4f}, "
#       f"Valid Acc={np.mean((y_p_v>0.5)==y_vld_f):.4f}")

In [15]:
# ============================================================
# Порог >= 50 + ANOVA order=1 + ALS
# ============================================================
import time

FREQ_THRESHOLD = 50

frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}

dataset_train_f = [
    row for row in dataset_train
    if all(p in frequent_players for p in row['players'])
]
dataset_valid_f = [
    row for row in dataset_valid_clean
    if all(p in frequent_players for p in row['players'])
]

print(f"Игроков: {len(frequent_players)}")
print(f"Train: {len(dataset_train_f)}, Valid: {len(dataset_valid_f)}")

# --- Индексация + RCM ---
freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
n_pl = len(pidx)

co = np.zeros((n_pl, n_pl), dtype=int)
for row in dataset_train_f:
    idxs = [pidx[p] for p in row['players']]
    for i in range(5):
        for j in range(i+1, 5):
            co[idxs[i], idxs[j]] += 1
            co[idxs[j], idxs[i]] += 1

rcm = reverse_cuthill_mckee(csr_matrix(co))

# --- Бинарные матрицы ---
I_trn_f = np.zeros((len(dataset_train_f), n_pl), dtype=int)
y_trn_f = np.zeros(len(dataset_train_f), dtype=float)
for i, row in enumerate(dataset_train_f):
    for p in row['players']:
        I_trn_f[i, pidx[p]] = 1
    y_trn_f[i] = row['y']
I_trn_f = I_trn_f[:, rcm]

I_vld_f = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
y_vld_f = np.zeros(len(dataset_valid_f), dtype=float)
for i, row in enumerate(dataset_valid_f):
    for p in row['players']:
        I_vld_f[i, pidx[p]] = 1
    y_vld_f[i] = row['y']
I_vld_f = I_vld_f[:, rcm]

good = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 1]
bad = [k for k in range(I_trn_f.shape[1]) if I_trn_f[:, k].max() == 0]
if len(bad) > 0:
    mask = np.ones(len(I_vld_f), dtype=bool)
    for k in bad:
        mask &= (I_vld_f[:, k] == 0)
    I_vld_f = I_vld_f[mask]
    y_vld_f = y_vld_f[mask]
I_trn_f = I_trn_f[:, good]
I_vld_f = I_vld_f[:, good]

print(f"\nI_trn: {I_trn_f.shape}, y=1: {y_trn_f.mean():.3f}")
print(f"I_vld: {I_vld_f.shape}, y=1: {y_vld_f.mean():.3f}")

# ============================================================
# ANOVA order=1
# ============================================================
print(f"\n{'='*50}")
print("ANOVA order=1, r=3...")
Y0 = teneva.anova(I_trn_f, y_trn_f, r=3, order=1)

y_p_t = teneva.get_many(Y0, I_trn_f)
y_p_v = teneva.get_many(Y0, I_vld_f)
trn_acc_anova = np.mean((y_p_t > 0.5) == y_trn_f)
vld_acc_anova = np.mean((y_p_v > 0.5) == y_vld_f)
print(f"  Train Acc={trn_acc_anova:.4f}, Valid Acc={vld_acc_anova:.4f}")

# ============================================================
# ALS
# ============================================================
print(f"\n{'='*50}")
print("ALS r=3, nswp=10, lamb=0.01...")

Y = teneva.als(
    I_trn_f, y_trn_f, Y0,
    nswp=10,
    I_vld=I_vld_f, y_vld=y_vld_f,
    lamb=0.01,
    log=True
)

y_p_t = teneva.get_many(Y, I_trn_f)
y_p_v = teneva.get_many(Y, I_vld_f)

trn_acc_als = np.mean((y_p_t > 0.5) == y_trn_f)
vld_acc_als = np.mean((y_p_v > 0.5) == y_vld_f)
trn_mae = np.mean(np.abs(y_p_t - y_trn_f))
vld_mae = np.mean(np.abs(y_p_v - y_vld_f))

print(f"\n{'='*50}")
print(f"ANOVA: Train Acc={trn_acc_anova:.4f}, Valid Acc={vld_acc_anova:.4f}")
print(f"ALS:   Train Acc={trn_acc_als:.4f}, Valid Acc={vld_acc_als:.4f}")
print(f"ALS:   Train MAE={trn_mae:.4f}, Valid MAE={vld_mae:.4f}")

Игроков: 408
Train: 2518, Valid: 111

I_trn: (2518, 407), y=1: 0.488
I_vld: (111, 407), y=1: 0.577

ANOVA order=1, r=3...
  Train Acc=0.6223, Valid Acc=0.4144

ALS r=3, nswp=10, lamb=0.01...
# pre | time:      0.010 | rank:   3.0 | e_vld: 1.0e+00 | 
#   1 | time:      0.624 | rank:   3.0 | e_vld: 1.1e+00 | e: 2.0e+28 | 
#   2 | time:      1.116 | rank:   3.0 | e_vld: 1.1e+00 | e: 3.1e+03 | 
#   3 | time:      1.604 | rank:   3.0 | e_vld: 1.3e+00 | e: 3.9e+01 | 
#   4 | time:      2.101 | rank:   3.0 | e_vld: 1.4e+00 | e: 1.2e+00 | 
#   5 | time:      2.593 | rank:   3.0 | e_vld: 1.4e+00 | e: 1.1e+00 | 
#   6 | time:      3.089 | rank:   3.0 | e_vld: 1.4e+00 | e: 1.0e+00 | 
#   7 | time:      3.584 | rank:   3.0 | e_vld: 1.4e+00 | e: 9.7e-01 | 
#   8 | time:      4.080 | rank:   3.0 | e_vld: 1.5e+00 | e: 9.0e-01 | 
#   9 | time:      4.579 | rank:   3.0 | e_vld: 1.5e+00 | e: 8.9e-01 | 
#  10 | time:      5.087 | rank:   3.0 | e_vld: 1.5e+00 | e: 8.5e-01 | stop: nswp | 

ANOVA: Train Acc

In [16]:
# ============================================================
# Построение 5-мерного тензора
# ============================================================

# Используем ВСЕ данные (без порога по частоте)
# Порог по частоте был нужен только для бинарного тензора

# --- Собираем всех игроков из train ---
all_players_train = set()
for row in dataset_train:
    for p in row['players']:
        all_players_train.add(p)

# Сортируем и строим индексацию
sorted_all = sorted(all_players_train)
player_to_idx5 = {pid: idx for idx, pid in enumerate(sorted_all)}
n_players5 = len(player_to_idx5)
print(f"Уникальных игроков в train: {n_players5}")

# --- Строим матрицу индексов (n_samples × 5) ---
def build_5d_matrix(dataset, p2idx):
    """
    Каждая пятёрка → 5 индексов (отсортированных по ID).
    """
    I = np.zeros((len(dataset), 5), dtype=int)
    y = np.zeros(len(dataset), dtype=float)
    
    for i, row in enumerate(dataset):
        # Сортируем игроков по ID → фиксированный порядок
        sorted_players = sorted(row['players'])
        for j, p in enumerate(sorted_players):
            I[i, j] = p2idx[p]
        y[i] = row['y']
    
    return I, y

I_trn5, y_trn5 = build_5d_matrix(dataset_train, player_to_idx5)

# Valid: оставляем только пятёрки, где все игроки есть в train
dataset_valid_5d = []
for row in dataset_valid_clean:
    all_known = True
    for p in row['players']:
        if p not in player_to_idx5:
            all_known = False
            break
    if all_known:
        dataset_valid_5d.append(row)

I_vld5, y_vld5 = build_5d_matrix(dataset_valid_5d, player_to_idx5)

print(f"\nI_trn: {I_trn5.shape}, y=1: {y_trn5.mean():.3f}")
print(f"I_vld: {I_vld5.shape}, y=1: {y_vld5.mean():.3f}")
print(f"\nПример строки I_trn: {I_trn5[0]}")
print(f"Мин индекс: {I_trn5.min()}, Макс индекс: {I_trn5.max()}")
print(f"Размерность каждой оси: {n_players5}")

# ============================================================
# ANOVA order=1 + ALS
# ============================================================
import time

print(f"\n{'='*60}")
print("ANOVA order=1, r=3...")
Y0 = teneva.anova(I_trn5, y_trn5, r=3, order=1)

y_p_t = teneva.get_many(Y0, I_trn5)
y_p_v = teneva.get_many(Y0, I_vld5)
trn_anova = np.mean((y_p_t > 0.5) == y_trn5)
vld_anova = np.mean((y_p_v > 0.5) == y_vld5)
print(f"  Train Acc={trn_anova:.4f}, Valid Acc={vld_anova:.4f}")

# --- ALS ---
print(f"\n{'='*60}")
print("ALS r=3, nswp=50, lamb=0.01...")
Y = teneva.als(
    I_trn5, y_trn5, Y0,
    nswp=50,
    I_vld=I_vld5, y_vld=y_vld5,
    lamb=0.01,
    log=True
)

y_p_t = teneva.get_many(Y, I_trn5)
y_p_v = teneva.get_many(Y, I_vld5)

trn_als = np.mean((y_p_t > 0.5) == y_trn5)
vld_als = np.mean((y_p_v > 0.5) == y_vld5)
trn_mae = np.mean(np.abs(y_p_t - y_trn5))
vld_mae = np.mean(np.abs(y_p_v - y_vld5))

print(f"\n{'='*60}")
print(f"Результаты (5-мерный тензор, {n_players5} игроков):")
print(f"  ANOVA: Train Acc={trn_anova:.4f}, Valid Acc={vld_anova:.4f}")
print(f"  ALS:   Train Acc={trn_als:.4f}, Valid Acc={vld_als:.4f}")
print(f"  ALS:   Train MAE={trn_mae:.4f}, Valid MAE={vld_mae:.4f}")
print(f"{'='*60}")

Уникальных игроков в train: 1126

I_trn: (9172, 5), y=1: 0.481
I_vld: (623, 5), y=1: 0.567

Пример строки I_trn: [  4 172 259 315 965]
Мин индекс: 0, Макс индекс: 1125
Размерность каждой оси: 1126

ANOVA order=1, r=3...


IndexError: index 549 is out of bounds for axis 1 with size 535

In [17]:
# ============================================================
# Построение 5-мерного тензора (исправленное)
# ============================================================

# --- Собираем всех игроков из train ---
all_players_train = set()
for row in dataset_train:
    for p in row['players']:
        all_players_train.add(p)

sorted_all = sorted(all_players_train)
player_to_idx5 = {pid: idx for idx, pid in enumerate(sorted_all)}
n_players5 = len(player_to_idx5)
print(f"Уникальных игроков в train: {n_players5}")

# --- Строим матрицу индексов (n_samples × 5) ---
def build_5d_matrix(dataset, p2idx):
    I = np.zeros((len(dataset), 5), dtype=int)
    y = np.zeros(len(dataset), dtype=float)
    for i, row in enumerate(dataset):
        sorted_players = sorted(row['players'])
        for j, p in enumerate(sorted_players):
            I[i, j] = p2idx[p]
        y[i] = row['y']
    return I, y

I_trn5, y_trn5 = build_5d_matrix(dataset_train, player_to_idx5)

dataset_valid_5d = []
for row in dataset_valid_clean:
    all_known = True
    for p in row['players']:
        if p not in player_to_idx5:
            all_known = False
            break
    if all_known:
        dataset_valid_5d.append(row)

I_vld5, y_vld5 = build_5d_matrix(dataset_valid_5d, player_to_idx5)

print(f"\nI_trn: {I_trn5.shape}, y=1: {y_trn5.mean():.3f}")
print(f"I_vld: {I_vld5.shape}, y=1: {y_vld5.mean():.3f}")

# Проверяем диапазон индексов
for ax in range(5):
    mn = I_trn5[:, ax].min()
    mx = I_trn5[:, ax].max()
    print(f"  Ось {ax}: min={mn}, max={mx}")

print(f"Размерность каждой оси: {n_players5}")

# ============================================================
# ANOVA + ALS
# ============================================================
import time

# --- ANOVA ---
print(f"\n{'='*60}")
print(f"ANOVA order=1, r=3 (оси размерности {n_players5})...")

# Добавляем фиктивную строку с максимальным индексом,
# чтобы ANOVA знала полный диапазон
max_idx = n_players5 - 1
I_trn5_ext = np.vstack([I_trn5, np.array([[max_idx]*5])])
y_trn5_ext = np.append(y_trn5, [0.5])

Y0 = teneva.anova(I_trn5_ext, y_trn5_ext, r=3, order=1)

# Проверяем размеры ядер
for k, Yk in enumerate(Y0):
    print(f"  Ядро {k}: shape={Yk.shape}")

# Предсказания
y_p_t = teneva.get_many(Y0, I_trn5)
y_p_v = teneva.get_many(Y0, I_vld5)
trn_anova = np.mean((y_p_t > 0.5) == y_trn5)
vld_anova = np.mean((y_p_v > 0.5) == y_vld5)
print(f"\n  Train Acc={trn_anova:.4f}, Valid Acc={vld_anova:.4f}")

# --- ALS ---
print(f"\n{'='*60}")
print("ALS r=3, nswp=50, lamb=0.01...")

Y = teneva.als(
    I_trn5, y_trn5, Y0,
    nswp=50,
    I_vld=I_vld5, y_vld=y_vld5,
    lamb=0.01,
    log=True
)

y_p_t = teneva.get_many(Y, I_trn5)
y_p_v = teneva.get_many(Y, I_vld5)

trn_als = np.mean((y_p_t > 0.5) == y_trn5)
vld_als = np.mean((y_p_v > 0.5) == y_vld5)
trn_mae = np.mean(np.abs(y_p_t - y_trn5))
vld_mae = np.mean(np.abs(y_p_v - y_vld5))

print(f"\n{'='*60}")
print(f"Результаты (5-мерный тензор, {n_players5} игроков):")
print(f"  ANOVA: Train Acc={trn_anova:.4f}, Valid Acc={vld_anova:.4f}")
print(f"  ALS:   Train Acc={trn_als:.4f}, Valid Acc={vld_als:.4f}")
print(f"  ALS:   Train MAE={trn_mae:.4f}, Valid MAE={vld_mae:.4f}")
print(f"{'='*60}")

Уникальных игроков в train: 1126

I_trn: (9172, 5), y=1: 0.481
I_vld: (623, 5), y=1: 0.567
  Ось 0: min=0, max=1024
  Ось 1: min=4, max=1031
  Ось 2: min=37, max=1079
  Ось 3: min=69, max=1120
  Ось 4: min=141, max=1125
Размерность каждой оси: 1126

ANOVA order=1, r=3 (оси размерности 1126)...
  Ядро 0: shape=(1, 536, 3)
  Ядро 1: shape=(3, 664, 3)
  Ядро 2: shape=(3, 733, 3)
  Ядро 3: shape=(3, 751, 3)
  Ядро 4: shape=(3, 690, 1)


IndexError: index 549 is out of bounds for axis 1 with size 536

In [18]:
# ============================================================
# Проверяем покрытие индексов по осям
# ============================================================

# Какие индексы реально используются на каждой оси?
for ax in range(5):
    used_trn = set(I_trn5[:, ax])
    used_vld = set(I_vld5[:, ax])
    all_used = used_trn | used_vld
    print(f"Ось {ax}: train [{min(used_trn)}..{max(used_trn)}], "
          f"уникальных={len(used_trn)}, "
          f"valid уникальных={len(used_vld)}, "
          f"только в valid={len(used_vld - used_trn)}")

# Проблема: индексы НЕ непрерывные (0,1,2,...),
# а разреженные (0, 5, 17, 549...)
# ANOVA создаёт ядро по макс. индексу в ОБУЧЕНИИ,
# но в данных есть индексы, которые она не видела

# --- Решение: переиндексация КАЖДОЙ оси отдельно ---
print(f"\nПереиндексация...")

# Собираем все индексы по каждой оси (из train)
axis_maps = []  # axis_maps[ax] = {старый_idx → новый_idx}

for ax in range(5):
    unique_vals = sorted(set(I_trn5[:, ax]))
    mapping = {old: new for new, old in enumerate(unique_vals)}
    axis_maps.append(mapping)
    print(f"  Ось {ax}: {len(unique_vals)} уникальных значений")

# --- Применяем переиндексацию к train ---
I_trn5r = np.zeros_like(I_trn5)
for i in range(len(I_trn5)):
    for ax in range(5):
        I_trn5r[i, ax] = axis_maps[ax][I_trn5[i, ax]]

# --- Применяем к valid (пропускаем строки с неизвестными индексами) ---
valid_mask = []
for i in range(len(I_vld5)):
    ok = True
    for ax in range(5):
        if I_vld5[i, ax] not in axis_maps[ax]:
            ok = False
            break
    valid_mask.append(ok)

valid_mask = np.array(valid_mask)
I_vld5_filtered = I_vld5[valid_mask]
y_vld5_filtered = y_vld5[valid_mask]

I_vld5r = np.zeros_like(I_vld5_filtered)
for i in range(len(I_vld5_filtered)):
    for ax in range(5):
        I_vld5r[i, ax] = axis_maps[ax][I_vld5_filtered[i, ax]]

print(f"\nПосле переиндексации:")
print(f"  I_trn: {I_trn5r.shape}")
print(f"  I_vld: {I_vld5r.shape} (было {len(I_vld5)})")
print(f"  y_trn mean: {y_trn5.mean():.3f}")
print(f"  y_vld mean: {y_vld5_filtered.mean():.3f}")

for ax in range(5):
    print(f"  Ось {ax}: размерность={I_trn5r[:, ax].max() + 1}")

# ============================================================
# ANOVA + ALS
# ============================================================
import time

print(f"\n{'='*60}")
print("ANOVA order=1, r=3...")
Y0 = teneva.anova(I_trn5r, y_trn5, r=3, order=1)

for k, Yk in enumerate(Y0):
    print(f"  Ядро {k}: shape={Yk.shape}")

y_p_t = teneva.get_many(Y0, I_trn5r)
y_p_v = teneva.get_many(Y0, I_vld5r)
trn_anova = np.mean((y_p_t > 0.5) == y_trn5)
vld_anova = np.mean((y_p_v > 0.5) == y_vld5_filtered)
print(f"  Train Acc={trn_anova:.4f}, Valid Acc={vld_anova:.4f}")

print(f"\n{'='*60}")
print("ALS r=3, nswp=50, lamb=0.01...")
Y = teneva.als(
    I_trn5r, y_trn5, Y0,
    nswp=50,
    I_vld=I_vld5r, y_vld=y_vld5_filtered,
    lamb=0.01,
    log=True
)

y_p_t = teneva.get_many(Y, I_trn5r)
y_p_v = teneva.get_many(Y, I_vld5r)

trn_als = np.mean((y_p_t > 0.5) == y_trn5)
vld_als = np.mean((y_p_v > 0.5) == y_vld5_filtered)
trn_mae = np.mean(np.abs(y_p_t - y_trn5))
vld_mae = np.mean(np.abs(y_p_v - y_vld5_filtered))

print(f"\n{'='*60}")
print(f"Результаты (5-мерный тензор):")
print(f"  ANOVA: Train Acc={trn_anova:.4f}, Valid Acc={vld_anova:.4f}")
print(f"  ALS:   Train Acc={trn_als:.4f}, Valid Acc={vld_als:.4f}")
print(f"  ALS:   Train MAE={trn_mae:.4f}, Valid MAE={vld_mae:.4f}")
print(f"{'='*60}")

Ось 0: train [0..1024], уникальных=535, valid уникальных=179, только в valid=9
Ось 1: train [4..1031], уникальных=663, valid уникальных=241, только в valid=19
Ось 2: train [37..1079], уникальных=732, valid уникальных=260, только в valid=20
Ось 3: train [69..1120], уникальных=750, valid уникальных=253, только в valid=14
Ось 4: train [141..1125], уникальных=690, valid уникальных=203, только в valid=7

Переиндексация...
  Ось 0: 535 уникальных значений
  Ось 1: 663 уникальных значений
  Ось 2: 732 уникальных значений
  Ось 3: 750 уникальных значений
  Ось 4: 690 уникальных значений

После переиндексации:
  I_trn: (9172, 5)
  I_vld: (524, 5) (было 623)
  y_trn mean: 0.481
  y_vld mean: 0.586
  Ось 0: размерность=535
  Ось 1: размерность=663
  Ось 2: размерность=732
  Ось 3: размерность=750
  Ось 4: размерность=690

ANOVA order=1, r=3...
  Ядро 0: shape=(1, 535, 3)
  Ядро 1: shape=(3, 663, 3)
  Ядро 2: shape=(3, 732, 3)
  Ядро 3: shape=(3, 750, 3)
  Ядро 4: shape=(3, 690, 1)
  Train Acc=0.6

In [21]:
# Проверяем столбцы pbp
pbp_file = os.path.join(pbp_dir, 'nhl_pbp20212022.csv')
pbp_check = pd.read_csv(pbp_file, nrows=100, low_memory=False)

# Ищем столбцы с позициями
pos_cols = [c for c in pbp_check.columns if 'pos' in c.lower()]
print(f"Столбцы с 'pos': {pos_cols}")

# Все столбцы с player/home/away
player_cols = [c for c in pbp_check.columns if 'layer' in c or 'home' in c.lower() or 'away' in c.lower()]
print(f"\nСтолбцы с игроками ({len(player_cols)}):")
for c in sorted(player_cols):
    val = pbp_check[c].dropna().iloc[0] if pbp_check[c].dropna().any() else 'NaN'
    print(f"  {c}: {val}")

Столбцы с 'pos': []

Столбцы с игроками (37):
  Away_Coach;: Mike Sullivan;
  Away_Goalie: TRISTAN JARRY
  Away_Goalie_Id: 8477465.0
  Away_Players: 6.0
  Away_Score: NaN
  Away_Team: PIT
  Home_Coach: Jon Cooper
  Home_Goalie: ANDREI VASILEVSKIY
  Home_Goalie_Id: 8476883.0
  Home_Players: 6.0
  Home_Score: NaN
  Home_Team: T.B
  Home_Zone: Neu
  awayPlayer1: JEFF CARTER
  awayPlayer1_id: 8470604.0
  awayPlayer2: BRYAN RUST
  awayPlayer2_id: 8475810.0
  awayPlayer3: DANTON HEINEN
  awayPlayer3_id: 8478046.0
  awayPlayer4: BRIAN DUMOULIN
  awayPlayer4_id: 8475208.0
  awayPlayer5: KRIS LETANG
  awayPlayer5_id: 8471724.0
  awayPlayer6: TRISTAN JARRY
  awayPlayer6_id: 8477465.0
  homePlayer1: BRAYDEN POINT
  homePlayer1_id: 8478010.0
  homePlayer2: NIKITA KUCHEROV
  homePlayer2_id: 8476453.0
  homePlayer3: ONDREJ PALAT
  homePlayer3_id: 8476292.0
  homePlayer4: RYAN MCDONAGH
  homePlayer4_id: 8474151.0
  homePlayer5: ERIK CERNAK
  homePlayer5_id: 8478416.0
  homePlayer6: ANDREI VASILEVSKIY

In [22]:
# ============================================================
# ALS с разной регуляризацией
# ============================================================

print(f"{'='*60}")
print(f"Подбор регуляризации ALS (r=3, nswp=50)")
print(f"{'='*60}")

# ANOVA baseline
Y0 = teneva.anova(I_trn5r, y_trn5, r=3, order=1)
y_p_v = teneva.get_many(Y0, I_vld5r)
vld_anova = np.mean((y_p_v > 0.5) == y_vld5_filtered)
print(f"ANOVA baseline: Valid Acc={vld_anova:.4f}\n")

for lamb in [0.1, 1.0, 10.0, 100.0, 1000.0]:
    Y0_copy = [G.copy() for G in Y0]
    
    Y = teneva.als(
        I_trn5r, y_trn5, Y0_copy,
        nswp=50,
        I_vld=I_vld5r, y_vld=y_vld5_filtered,
        lamb=lamb,
        log=False
    )
    
    y_p_t = teneva.get_many(Y, I_trn5r)
    y_p_v = teneva.get_many(Y, I_vld5r)
    
    trn_acc = np.mean((y_p_t > 0.5) == y_trn5)
    vld_acc = np.mean((y_p_v > 0.5) == y_vld5_filtered)
    
    print(f"  lamb={lamb:8.1f}: Train Acc={trn_acc:.4f}, Valid Acc={vld_acc:.4f}")

Подбор регуляризации ALS (r=3, nswp=50)
ANOVA baseline: Valid Acc=0.4809

  lamb=     0.1: Train Acc=0.9989, Valid Acc=0.4504
  lamb=     1.0: Train Acc=0.9070, Valid Acc=0.4599
  lamb=    10.0: Train Acc=0.5193, Valid Acc=0.4141
  lamb=   100.0: Train Acc=0.5193, Valid Acc=0.4141
  lamb=  1000.0: Train Acc=0.5193, Valid Acc=0.4141


In [29]:
# ============================================================
# [Address20] тензор, d ~ 100-150 игроков
# ============================================================

# Порог >= 80: 213 игроков, 508 пятёрок
# Порог >= 90: 151 игрок, 114 пятёрок — мало
# Попробуем >= 80

FREQ_THRESHOLD = 80

frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}

dataset_train_f = [
    row for row in dataset_train
    if all(p in frequent_players for p in row['players'])
]
dataset_valid_f = [
    row for row in dataset_valid_clean
    if all(p in frequent_players for p in row['players'])
]

print(f"Порог >= {FREQ_THRESHOLD}:")
print(f"  [Surname68]: {len(frequent_players)}")
print(f"  Train: {len(dataset_train_f)}, Valid: {len(dataset_valid_f)}")

if len(dataset_valid_f) == 0:
    print("  ⚠️ Valid пуст!")

# --- Индексация + RCM ---
freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
n_pl = len(pidx)

co = np.zeros((n_pl, n_pl), dtype=int)
for row in dataset_train_f:
    idxs = [pidx[p] for p in row['players']]
    for i in range(5):
        for j in range(i+1, 5):
            co[idxs[i], idxs[j]] += 1
            co[idxs[j], idxs[i]] += 1
rcm = reverse_cuthill_mckee(csr_matrix(co))

# --- Бинарные матрицы ---
I_trn_b = np.zeros((len(dataset_train_f), n_pl), dtype=int)
y_trn_b = np.zeros(len(dataset_train_f), dtype=float)
for i, row in enumerate(dataset_train_f):
    for p in row['players']:
        I_trn_b[i, pidx[p]] = 1
    y_trn_b[i] = row['y']
I_trn_b = I_trn_b[:, rcm]

I_vld_b = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
y_vld_b = np.zeros(len(dataset_valid_f), dtype=float)
for i, row in enumerate(dataset_valid_f):
    for p in row['players']:
        I_vld_b[i, pidx[p]] = 1
    y_vld_b[i] = row['y']
I_vld_b = I_vld_b[:, rcm]

good = [k for k in range(I_trn_b.shape[1]) if I_trn_b[:, k].max() == 1]
bad = [k for k in range(I_trn_b.shape[1]) if I_trn_b[:, k].max() == 0]
if len(bad) > 0:
    mask = np.ones(len(I_vld_b), dtype=bool)
    for k in bad:
        mask &= (I_vld_b[:, k] == 0)
    I_vld_b = I_vld_b[mask]
    y_vld_b = y_vld_b[mask]
I_trn_b = I_trn_b[:, good]
I_vld_b = I_vld_b[:, good]

print(f"\n  I_trn: {I_trn_b.shape}, y=1: {y_trn_b.mean():.3f}")
print(f"  I_vld: {I_vld_b.shape}, y=1: {y_vld_b.mean():.3f}" if len(y_vld_b) > 0 else "  I_vld: пуст")

# ============================================================
# ANOVA order=1 + ALS с подбором регуляризации
# ============================================================

if len(y_vld_b) > 0:
    print(f"\n{'='*60}")
    
    Y0 = teneva.anova(I_trn_b, y_trn_b, r=2, order=1)
    y_p_t = [Surname69]eva.get_many(Y0, I_trn_b)
    y_p_v = [Surname70].get_many(Y0, I_vld_b)
    trn_a = np.mean((y_p_t > 0.5) == y_trn_b)
    vld_a = np.mean((y_p_v > 0.5) == y_vld_b)
    print(f"ANOVA r=2: Train={trn_a:.4f}, Valid={vld_a:.4f}")
    
    print(f"\nALS подбор регуляризации (r=2, nswp=20):")
    for lamb in [0.01, 0.1, 1.0, 10.0, 100.0]:
        Y0_copy = [G.copy() for G in Y0]
        Y = teneva.als(
            I_trn_b, y_trn_b, Y0_copy,
            nswp=20, lamb=lamb, log=False
        )
        y_p_t = teneva.get_many(Y, I_trn_b)
        y_p_v = [Surname71]eva.get_many(Y, I_vld_b)
        trn_acc = np.mean((y_p_t > 0.5) == y_trn_b)
        vld_acc = np.mean((y_p_v > 0.5) == y_vld_b)
        print(f"  lamb={lamb:7.2f}: Train={trn_acc:.4f}, Valid={vld_acc:.4f}")

SyntaxError: invalid syntax (3434009882.py, line 82)

In [26]:
# ============================================================
# ANOVA order=1 + ALS с подбором регуляризации
# ============================================================

if len(y_vld_b) > 0:
    print(f"\n{'='*60}")
    
    Y0 = teneva.anova(I_trn_b, y_trn_b, r=2, order=1)
    y_p_t = teneva.get_many(Y0, I_trn_b)
    y_p_v = teneva.get_many(Y0, I_vld_b)
    trn_a = np.mean((y_p_t > 0.5) == y_trn_b)
    vld_a = np.mean((y_p_v > 0.5) == y_vld_b)
    print(f"ANOVA r=2: Train={trn_a:.4f}, Valid={vld_a:.4f}")
    
    print(f"\nALS подбор регуляризации (r=2, nswp=20):")
    for lamb in [0.01, 0.1, 1.0, 10.0, 100.0]:
        Y0_copy = [G.copy() for G in Y0]
        Y = teneva.als(
            I_trn_b, y_trn_b, Y0_copy,
            nswp=20, lamb=lamb, log=False
        )
        y_p_t = teneva.get_many(Y, I_trn_b)
        y_p_v = teneva.get_many(Y, I_vld_b)
        trn_acc = np.mean((y_p_t > 0.5) == y_trn_b)
        vld_acc = np.mean((y_p_v > 0.5) == y_vld_b)
        print(f"  lamb={lamb:7.2f}: Train={trn_acc:.4f}, Valid={vld_acc:.4f}")

NameError: name 'y_vld_b' is not defined

In [27]:
y_p_t

array([0., 0., 0., ..., 0., 0., 0.])

In [28]:
# ============================================================
# ANOVA + ALS (бинарный тензор, порог >= 80)
# ============================================================

print(f"I_trn: {I_trn_b.shape}, y=1: {y_trn_b.mean():.3f}")
print(f"I_vld: {I_vld_b.shape}, y=1: {y_vld_b.mean():.3f}")

print(f"\n{'='*60}")
Y0 = teneva.anova(I_trn_b, y_trn_b, r=2, order=1)
y_p_t = teneva.get_many(Y0, I_trn_b)
y_p_v = teneva.get_many(Y0, I_vld_b)
print(f"ANOVA r=2: Train={np.mean((y_p_t>0.5)==y_trn_b):.4f}, "
      f"Valid={np.mean((y_p_v>0.5)==y_vld_b):.4f}")

print(f"\nALS подбор регуляризации (r=2, nswp=20):")
for lamb in [0.01, 0.1, 1.0, 10.0, 100.0]:
    Y0_copy = [G.copy() for G in Y0]
    Y = teneva.als(I_trn_b, y_trn_b, Y0_copy,
                   nswp=20, lamb=lamb, log=False)
    y_p_t = teneva.get_many(Y, I_trn_b)
    y_p_v = teneva.get_many(Y, I_vld_b)
    trn_acc = np.mean((y_p_t > 0.5) == y_trn_b)
    vld_acc = np.mean((y_p_v > 0.5) == y_vld_b)
    print(f"  lamb={lamb:7.2f}: Train={trn_acc:.4f}, Valid={vld_acc:.4f}")

NameError: name 'I_trn_b' is not defined

In [30]:
# ============================================================
# Бинарный тензор, порог >= 80
# ============================================================

FREQ_THRESHOLD = 80
frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}

dataset_train_f = [
    row for row in dataset_train
    if all(p in frequent_players for p in row['players'])
]
dataset_valid_f = [
    row for row in dataset_valid_clean
    if all(p in frequent_players for p in row['players'])
]

print(f"Порог >= {FREQ_THRESHOLD}:")
print(f"  Игроков: {len(frequent_players)}")
print(f"  Train: {len(dataset_train_f)}, Valid: {len(dataset_valid_f)}")

freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
n_pl = len(pidx)

co = np.zeros((n_pl, n_pl), dtype=int)
for row in dataset_train_f:
    idxs = [pidx[p] for p in row['players']]
    for i in range(5):
        for j in range(i+1, 5):
            co[idxs[i], idxs[j]] += 1
            co[idxs[j], idxs[i]] += 1
rcm = reverse_cuthill_mckee(csr_matrix(co))

I_trn_b = np.zeros((len(dataset_train_f), n_pl), dtype=int)
y_trn_b = np.zeros(len(dataset_train_f), dtype=float)
for i, row in enumerate(dataset_train_f):
    for p in row['players']:
        I_trn_b[i, pidx[p]] = 1
    y_trn_b[i] = row['y']
I_trn_b = I_trn_b[:, rcm]

I_vld_b = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
y_vld_b = np.zeros(len(dataset_valid_f), dtype=float)
for i, row in enumerate(dataset_valid_f):
    for p in row['players']:
        I_vld_b[i, pidx[p]] = 1
    y_vld_b[i] = row['y']
I_vld_b = I_vld_b[:, rcm]

good = [k for k in range(I_trn_b.shape[1]) if I_trn_b[:, k].max() == 1]
bad = [k for k in range(I_trn_b.shape[1]) if I_trn_b[:, k].max() == 0]
if len(bad) > 0:
    mask = np.ones(len(I_vld_b), dtype=bool)
    for k in bad:
        mask &= (I_vld_b[:, k] == 0)
    I_vld_b = I_vld_b[mask]
    y_vld_b = y_vld_b[mask]
I_trn_b = I_trn_b[:, good]
I_vld_b = I_vld_b[:, good]

print(f"\nI_trn: {I_trn_b.shape}, y=1: {y_trn_b.mean():.3f}")
if len(y_vld_b) > 0:
    print(f"I_vld: {I_vld_b.shape}, y=1: {y_vld_b.mean():.3f}")
else:
    print("I_vld: пуст!")

Порог >= 80:
  Игроков: 213
  Train: 508, Valid: 28

I_trn: (508, 182), y=1: 0.476
I_vld: (25, 182), y=1: 0.480


In [31]:
# ============================================================
# ANOVA + ALS
# ============================================================

Y0 = teneva.anova(I_trn_b, y_trn_b, r=2, order=1)
y_p_t = teneva.get_many(Y0, I_trn_b)
y_p_v = teneva.get_many(Y0, I_vld_b)
print(f"ANOVA r=2: Train={np.mean((y_p_t>0.5)==y_trn_b):.4f}, "
      f"Valid={np.mean((y_p_v>0.5)==y_vld_b):.4f}")

print(f"\nALS подбор регуляризации (r=2, nswp=20):")
for lamb in [0.01, 0.1, 1.0, 10.0, 100.0]:
    Y0_copy = [G.copy() for G in Y0]
    Y = teneva.als(I_trn_b, y_trn_b, Y0_copy,
                   nswp=20, lamb=lamb, log=False)
    y_p_t = teneva.get_many(Y, I_trn_b)
    y_p_v = teneva.get_many(Y, I_vld_b)
    trn_acc = np.mean((y_p_t > 0.5) == y_trn_b)
    vld_acc = np.mean((y_p_v > 0.5) == y_vld_b)
    print(f"  lamb={lamb:7.2f}: Train={trn_acc:.4f}, Valid={vld_acc:.4f}")

ANOVA r=2: Train=0.6417, Valid=0.2800

ALS подбор регуляризации (r=2, nswp=20):
  lamb=   0.01: Train=0.9449, Valid=0.4400
  lamb=   0.10: Train=0.9291, Valid=0.4000
  lamb=   1.00: Train=0.7205, Valid=0.3200
  lamb=  10.00: Train=0.5236, Valid=0.5200
  lamb= 100.00: Train=0.5236, Valid=0.5200


In [32]:
# ============================================================
# Порог >= 20 (максимум данных) + ANOVA order=1 + ALS
# ============================================================

FREQ_THRESHOLD = 20
frequent_players = {pid for pid, freq in player_freq.items() if freq >= FREQ_THRESHOLD}

dataset_train_f = [
    row for row in dataset_train
    if all(p in frequent_players for p in row['players'])
]
dataset_valid_f = [
    row for row in dataset_valid_clean
    if all(p in frequent_players for p in row['players'])
]

freq_sorted = sorted(frequent_players, key=lambda p: -player_freq[p])
pidx = {pid: idx for idx, pid in enumerate(freq_sorted)}
n_pl = len(pidx)

co = np.zeros((n_pl, n_pl), dtype=int)
for row in dataset_train_f:
    idxs = [pidx[p] for p in row['players']]
    for i in range(5):
        for j in range(i+1, 5):
            co[idxs[i], idxs[j]] += 1
            co[idxs[j], idxs[i]] += 1
rcm = reverse_cuthill_mckee(csr_matrix(co))

I_trn_b = np.zeros((len(dataset_train_f), n_pl), dtype=int)
y_trn_b = np.zeros(len(dataset_train_f), dtype=float)
for i, row in enumerate(dataset_train_f):
    for p in row['players']:
        I_trn_b[i, pidx[p]] = 1
    y_trn_b[i] = row['y']
I_trn_b = I_trn_b[:, rcm]

I_vld_b = np.zeros((len(dataset_valid_f), n_pl), dtype=int)
y_vld_b = np.zeros(len(dataset_valid_f), dtype=float)
for i, row in enumerate(dataset_valid_f):
    for p in row['players']:
        I_vld_b[i, pidx[p]] = 1
    y_vld_b[i] = row['y']
I_vld_b = I_vld_b[:, rcm]

good = [k for k in range(I_trn_b.shape[1]) if I_trn_b[:, k].max() == 1]
bad = [k for k in range(I_trn_b.shape[1]) if I_trn_b[:, k].max() == 0]
if len(bad) > 0:
    mask = np.ones(len(I_vld_b), dtype=bool)
    for k in bad:
        mask &= (I_vld_b[:, k] == 0)
    I_vld_b = I_vld_b[mask]
    y_vld_b = y_vld_b[mask]
I_trn_b = I_trn_b[:, good]
I_vld_b = I_vld_b[:, good]

print(f"Порог >= {FREQ_THRESHOLD}")
print(f"I_trn: {I_trn_b.shape}, y=1: {y_trn_b.mean():.3f}")
print(f"I_vld: {I_vld_b.shape}, y=1: {y_vld_b.mean():.3f}")

Порог >= 20
I_trn: (6536, 658), y=1: 0.484
I_vld: (308, 658), y=1: 0.568


In [33]:
# ============================================================
# ANOVA order=1
# ============================================================

Y0 = teneva.anova(I_trn_b, y_trn_b, r=2, order=1)
y_p_t = teneva.get_many(Y0, I_trn_b)
y_p_v = teneva.get_many(Y0, I_vld_b)
trn_anova = np.mean((y_p_t > 0.5) == y_trn_b)
vld_anova = np.mean((y_p_v > 0.5) == y_vld_b)
print(f"ANOVA r=2: Train={trn_anova:.4f}, Valid={vld_anova:.4f}")

# ============================================================
# ALS: подбор ранга и регуляризации
# ============================================================

print(f"\n{'='*60}")
print(f"ALS подбор гиперпараметров (nswp=20):")
print(f"{'='*60}")

best_vld = 0
best_params = {}

for r in [2, 3, 5]:
    Y0r = teneva.anova(I_trn_b, y_trn_b, r=r, order=1)
    
    for lamb in [0.01, 0.1, 1.0, 10.0, 100.0]:
        Y0_copy = [G.copy() for G in Y0r]
        Y = teneva.als(
            I_trn_b, y_trn_b, Y0_copy,
            nswp=20, lamb=lamb, log=False
        )
        y_p_t = teneva.get_many(Y, I_trn_b)
        y_p_v = teneva.get_many(Y, I_vld_b)
        trn_acc = np.mean((y_p_t > 0.5) == y_trn_b)
        vld_acc = np.mean((y_p_v > 0.5) == y_vld_b)
        
        marker = ""
        if vld_acc > best_vld:
            best_vld = vld_acc
            best_params = {'r': r, 'lamb': lamb}
            marker = " ★"
        
        print(f"  r={r}, lamb={lamb:7.2f}: "
              f"Train={trn_acc:.4f}, Valid={vld_acc:.4f}{marker}")

print(f"\n{'='*60}")
print(f"Лучшие параметры: r={best_params['r']}, lamb={best_params['lamb']}")
print(f"Лучший Valid Acc: {best_vld:.4f}")
print(f"ANOVA baseline:  {vld_anova:.4f}")
print(f"{'='*60}")

ANOVA r=2: Train=0.6102, Valid=0.4610

ALS подбор гиперпараметров (nswp=20):
  r=2, lamb=   0.01: Train=0.8442, Valid=0.4968 ★
  r=2, lamb=   0.10: Train=0.8345, Valid=0.5292 ★
  r=2, lamb=   1.00: Train=0.6799, Valid=0.4773
  r=2, lamb=  10.00: Train=0.6152, Valid=0.4091
  r=2, lamb= 100.00: Train=0.5158, Valid=0.4318
  r=3, lamb=   0.01: Train=0.9353, Valid=0.4805
  r=3, lamb=   0.10: Train=0.9132, Valid=0.5000
  r=3, lamb=   1.00: Train=0.6799, Valid=0.4773
  r=3, lamb=  10.00: Train=0.6152, Valid=0.4091
  r=3, lamb= 100.00: Train=0.5158, Valid=0.4318
  r=5, lamb=   0.01: Train=0.9832, Valid=0.4838
  r=5, lamb=   0.10: Train=0.9370, Valid=0.4708
  r=5, lamb=   1.00: Train=0.6799, Valid=0.4773
  r=5, lamb=  10.00: Train=0.6152, Valid=0.4091
  r=5, lamb= 100.00: Train=0.5158, Valid=0.4318

Лучшие параметры: r=2, lamb=0.1
Лучший Valid Acc: 0.5292
ANOVA baseline:  0.4610


In [34]:
# Проверим, есть ли у нас другие файлы данных
for f in os.listdir(pbp_dir):
    print(f)

nhl_pbp20132014.csv
nhl_pbp20152016.csv
nhl_pbp20172018.csv
nhl_pbp20122013.csv
nhl_pbp20112012.csv
nhl_pbp20102011.csv
nhl_pbp20182019.csv
nhl_pbp20092010.csv
nhl_pbp20222023.csv
nhl_pbp20212022.csv
nhl_pbp20082009.csv
.ipynb_checkpoints
nhl_pbp20142015.csv
nhl_pbp20202021.csv
nhl_pbp20072008.csv
nhl_pbp20162017.csv
nhl_pbp20192020.csv



📋 Итоги работы с данными NHL PBP (play-by-play)

Данные
Использовались play-by-play данные NHL с Kaggle (nhl_pbp{season}.csv) за сезоны 2017-2022. Из событий вбрасываний (FAC) извлекались составы пятёрок на льду, а по изменению счёта считались голы за (GF) и голы против (GA) для каждой пятёрки.

Предобработка
Извлечение пятёрок : из каждого матча по событиям FAC определялись интервалы с фиксированным составом (5 полевых игроков), подсчитывалось время на льду и голы.
Агрегация : пятёрки агрегировались по сезонам с взвешиванием (более поздние сезоны получали больший вес). Фильтрация: минимум 120 секунд на льду, минимум 5 игр, хотя бы 1 гол (за или против).
Целевая переменная : бинарная — y = 1 если GF > GA (пятёрка забила больше, чем пропустила), иначе y = 0.
Train/Valid split : сезоны 2017-2021 → train, сезон 2022 → valid.
Подходы к тензорному представлению
1. Бинарный тензор (d осей × 2 значения)
Каждый игрок — отдельная ось тензора (0 = не на льду, 1 = на льду). Размерность d = число уникальных игроков.

ANOVA order=1 : работает при любом d, но это линейная модель — не учитывает взаимодействия между игроками. Результат: Train Acc ≈ 61%, Valid Acc ≈ 46%.
ANOVA order=2 : должна учитывать парные взаимодействия, но вычисление O(d²) пар. При d > 200 ядро Jupyter падает (out of memory). Не удалось запустить ни при каком пороге частоты.
ALS (Alternating Least Squares) : стартуя с ANOVA order=1, итеративно оптимизирует TT-ядра. При d > 500 — числовой взрыв (значения до 10⁶⁶). При d ≈ 200 с регуляризацией — сильное переобучение: Train 94%, Valid 44%. При увеличении регуляризации модель схлопывается до предсказания константы.
Лучший результат бинарного тензора : порог ≥ 20 (658 игроков, 6536 train, 308 valid), ALS r=2, λ=0.1 → Train 83%, Valid 52.9% .
2. 5-мерный тензор (5 осей × N игроков)
Каждая ось — позиция в пятёрке (1-5), значение — ID игрока. Игроки в пятёрке отсортированы по ID для фиксации порядка.

Проблема : один и тот же игрок на разных осях воспринимается моделью как разные сущности. При сортировке по ID возникает искусственная зависимость от позиции.
ANOVA order=1 : Train 70%, Valid 48%.
ALS : полное переобучение — Train 99.9%, Valid 49%. Регуляризация не помогает.
Ограничения данных PBP
Голы — редкое событие : 5-6 голов за матч на обе команды, пятёрка играет 2-3 минуты → большинство интервалов без голов. Бинарная метрика GF > GA очень шумная.
В PBP файлах отсутствуют события бросков (SHOT, MISS, BLOCK, GOAL). Доступны только: FAC, STOP, PSTR, PEND, DELPEN, GEND, CHL, SOC. Это не позволяет вычислить метрику Corsi (броски за/против), которая значительно информативнее голов.
Нет информации о позициях игроков (C, LW, RW, D) — невозможно построить осмысленный 5-мерный тензор с позиционными осями.
Разреженность : при жёстких порогах частоты (для уменьшения d) объём valid данных падает до 25-100 наблюдений, что делает оценку ненадёжной.
Вывод и дальнейшие шаги
Данные PBP с Kaggle оказались недостаточными для задачи моделирования совместимости:

Метрика на основе голов слишком шумная,
Отсутствие бросковой статистики не позволяет использовать Corsi/Fenwick,
Бинарный тензор при большом d не поддаётся TT-разложению с взаимодействиями.
Решение : переход на данные MoneyPuck (lines.csv), которые содержат:

Готовые составы троек нападающих и пар защитников,
Метрики Corsi%, xGoals%, Fenwick% — значительно более информативные,
Данные на уровне отдельных игр за сезоны 2008-2024.
Это позволит построить 3-мерный тензор (тройки нападающих) с непрерывной целевой переменной (CF% или xGF%), что должно дать существенно лучшие результаты.

